In [ ]:
# Cell 1: Install dependencies (Ensure your venv is active)
!pip install --upgrade pip
!pip install vllm lmcache

# Cell 2: Configure the 3-Tier Caching System
import os
import yaml

# Define the multi-tier hierarchy: GPU (Tier 1) -> CPU RAM (Tier 2) -> SSD (Tier 3)
lmcache_config = {
    "chunk_size": 256,
    "local_cpu": True,
    "max_local_cpu_size": 2.0,      # Tier 2: 2 GB of system RAM
    "local_disk": "/tmp/lmcache",   # Tier 3: Disk storage path
    "max_local_disk_size": 10.0     # Tier 3: 10 GB of disk space
}

# Write the configuration to a YAML file
with open("lmcache_config.yaml", "w") as f:
    yaml.dump(lmcache_config, f)

# Point LMCache to this configuration BEFORE importing vLLM
os.environ["LMCACHE_CONFIG_FILE"] = "lmcache_config.yaml"

In [2]:
from lmcache_vllm.vllm_wrapper import LMCacheVLLMStrategy
from vllm import LLM, SamplingParams

# Step 1: Define the model name (TinyML model for 6GB VRAM)
model_name = "meta-llama/Llama-3.2-1B"

# Step 2: Initialize the LMCache Strategy
# This tells vLLM to use LMCache's logic for handling KV caches
lmcache_strategy = LMCacheVLLMStrategy()

# Step 3: Initialize the vLLM engine with LMCache integration
# gpu_memory_utilization is set to 0.7 to save room for the cache and OS
llm = LLM(
    model=model_name,
    gpu_memory_utilization=0.7, 
    trust_remote_code=True,
    kv_cache_strategy=lmcache_strategy 
)

# Step 4: Set sampling parameters (standard for testing)
sampling_params = SamplingParams(temperature=0, max_tokens=100)

ModuleNotFoundError: No module named 'lmcache_vllm'